# Xây dựng Knowledge Base dùng chung

Audit XIV/XVI/XVII + GIR + TT85 → metadata namespace/chunk_type → BGE-M3 dense/sparse → PostgreSQL/pgvector → graph và kiểm chứng. Hai namespace dùng cùng vector store. Biểu mẫu TT85 chỉ nằm trong danh mục template, không embedding. Logic dùng chung trong rag/; kết quả giữ tại output notebook. Hiệu lực pháp lý là metadata nguồn local, chưa xác minh độc lập trong workflow này.


In [1]:
import os
import sys
from pathlib import Path

candidates = [Path(os.environ["AIAGENT_ROOT"])] if "AIAGENT_ROOT" in os.environ else []
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([parent, parent / "apps/aiAgent", parent / "frappe-bench/apps/aiAgent"])
ROOT = next((p.resolve() for p in candidates if (p / "rag/data.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set AIAGENT_ROOT to the absolute aiAgent folder")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from rag.data import config  # noqa: E402 - project root must be resolved first

cfg = config()
print("Workspace:", ROOT)

Workspace: /Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent


In [2]:
import time

import numpy as np
import pandas as pd
from IPython.display import display
from IPython.utils.io import capture_output

from rag.data import audit, build_corpus
from rag.db import connect, counts, ingest, migrate
from rag.embedding import Embedder, load_embeddings

## Đối chiếu nguồn và sửa đầu vào tìm kiếm — 27/09/2026
Đã xem trực tiếp PDF địa phương và đối chiếu Công báo TT31/WCO HS2022. Sửa khoảng trắng 87.04; giữ nguyên tên pháp lý 85.06/85.34, bổ sung ngữ cảnh tiếng Việt có dẫn nguồn và Chú giải 8 có citation cho search text. Raw và graph giữ nguyên. Đây là thay đổi dữ liệu được cho phép, không triển khai các đề xuất pipeline đang chờ xem xét. Cell chỉ sửa đầu vào; snapshot active và benchmark trước đó vẫn thuộc phiên bản cũ cho đến khi chạy lại các bước ingestion và đánh giá.
Search views của XIV/XVII được dựng lại từ trường tiếng Việt trong master; bản gốc song ngữ vẫn giữ để truy vết.


In [3]:
# Authorized source-only corrections, verified against local PDFs and official publications.
import copy
import hashlib
import json
from pathlib import Path

import jsonschema

from rag.data import NORMALIZED, ROOT, SOURCES, read, xiv_vietnamese_view, xvii_vietnamese_view

OLD_8704 = 'Xecó độngcơdùng đểchởhàng.'
VERIFIED_8704 = 'Xe có động cơ dùng để chở hàng.'
VIETNAM_8704_URL = 'https://congbaocdn.chinhphu.vn/CongBaoCP/VanBan/2022/6/37431/41050-1-2022553-55431-2022-tt-btc.pdf'
WCO_85_URL = 'https://www.wcoomd.org/-/media/wco/public/global/pdf/topics/nomenclature/instruments-and-tools/hs-nomenclature-2022/2022/1685_2022e.pdf?la=en'
verified_on = '2026-09-27'
source_evidence = [
    {'target': '87.04', 'local_pdf': 'data/raw/TT31-Section-XVII.pdf', 'pdf_page': 50,
     'printed_page': '1003', 'official_url': VIETNAM_8704_URL, 'official_pdf_page': 58,
     'decision': 'Whitespace correction only; original extracted raw text retained'},
    {'target': '85.06', 'local_pdf': 'data/raw/TT_31-Section-XVI.pdf', 'pdf_page': 147,
     'printed_page': '904', 'official_url': WCO_85_URL, 'official_pdf_page': 6,
     'decision': 'Keep the verified Vietnamese heading; exclude English from retrieval text'},
    {'target': '85.34 / Chapter 85 Note 8', 'local_pdf': 'data/raw/TT_31-Section-XVI.pdf',
     'pdf_page': '179 / 122', 'printed_page': '936 / 886', 'official_url': WCO_85_URL,
     'official_pdf_page': '14 / 2',
     'decision': 'Legal heading retained; add complete source Note 8 as explicitly cited search context'},
]
for evidence in source_evidence:
    evidence['local_pdf_sha256'] = hashlib.sha256((ROOT / evidence['local_pdf']).read_bytes()).hexdigest()
paths = {k: NORMALIZED / SOURCES[k] for k in (
    'tt31_section_xiv_master', 'tt31_section_xiv',
    'tt31_section_xvi_master', 'tt31_section_xvi',
    'tt31_section_xvii_master', 'tt31_section_xvii')}
data = {k: read(p) for k, p in paths.items()}
original = copy.deepcopy(data)
masters = {r['record_id']: r for r in data['tt31_section_xvi_master']}
assert masters['SXVI-T-8506']['text_vi'] == 'Pin và bộ pin.'
assert masters['SXVI-T-8506']['text_en'] == 'Primary cells and primary batteries.'
assert masters['SXVI-T-8534']['text_vi'] == 'Mạch in.'
assert masters['SXVI-T-8534']['text_en'] == 'Printed circuits.'
note = masters['SXVI-N-CH85_NOTE_8']
assert note['text_vi'].startswith('Theo mục đích của nhóm 85.34')
assert 'phần tử kết nối' in note['text_vi'] and '85.42' in note['text_vi']
texts = {
    'SXVI-T-8506': 'Mã 85.06; Pin và bộ pin.',
    'SXVI-T-8534': 'Mã 85.34; Mạch in.\nNgữ cảnh nguồn: Chương 85 — Chú giải 8 [SXVI-N-CH85_NOTE_8; trang PDF 122].\n'
        + note['text_vi'],
}
for sid, text in texts.items():
    m = masters[sid]
    m['retrieval']['embedding_text_vi'] = text
    m['retrieval']['lexical_text_vi'] = text
    m['retrieval']['source_verification'] = {'verified_on': verified_on, 'official_url': WCO_85_URL,
        'basis': 'Vietnamese statutory heading and cited Chapter 85 Note 8; no generated legal paraphrase'}
    if sid == 'SXVI-T-8534':
        m['context_source_ids'] = [sid, note['record_id']]
        m['retrieval']['source_context'] = [{'source_record_id': note['record_id'], 'locator': note['source']}]
for row in data['tt31_section_xvi']:
    if row['id'] in texts:
        row['text'] = row['sparse_text'] = texts[row['id']]
        row['metadata']['retrieval_source_verified_on'] = verified_on
        if row['id'] == 'SXVI-T-8534':
            row['metadata']['context_source_ids'] = [row['id'], note['record_id']]

# Correct five bilingual OCR spills using the Vietnamese columns of TT31 Section XVI.
# Page numbers below are PDF pages (printed pages are PDF page + 757).
verified_xvi_texts = {
    'SXVI-T-85362099': ('Loại khác', 181, 'Of a kind used in radio equipment'),
    'SXVI-T-85369039': ('Loại khác', 184, 'Battery clamps, of a kind'),
    'SXVI-T-8538': ('Bộ phận chỉ dùng hoặc chủ yếu dùng với các thiết bị thuộc nhóm 85.35, 85.36 hoặc 85.37.',
                    185, 'with the apparatus of heading'),
    'SXVI-T-85389012': ('Bộ phận của hàng hoá thuộc phân nhóm 8536.50.51, 8536.50.59, 8536.69.32 hoặc 8536.69.39',
                       186, 'or 8536.69.39'),
    'SXVI-T-85491310': ('Của pin và bộ pin', 199, 'Of electric accumulators of a kind'),
}
xvi_source_url = 'https://congbao.chinhphu.vn/van-ban/thong-tu-so-31-2022-tt-btc-37431/41047.htm'
for sid, (vi, pdf_page, spill) in verified_xvi_texts.items():
    m = masters[sid]
    assert m['source']['page_start'] == pdf_page
    previous = m['text_vi']
    assert previous == vi or spill in previous, (sid, previous)
    if previous != vi:
        for descendant in data['tt31_section_xvi_master']:
            for field in ('embedding_text_vi', 'lexical_text_vi'):
                value = descendant['retrieval'].get(field)
                if isinstance(value, str):
                    descendant['retrieval'][field] = value.replace(previous, vi)
        for row in data['tt31_section_xvi']:
            for field in ('text', 'sparse_text'):
                row[field] = row[field].replace(previous, vi)
        m['text_vi'] = vi
    m['retrieval']['source_verification'] = {
        'verified_on': verified_on,
        'local_pdf': 'data/raw/TT_31-Section-XVI.pdf',
        'local_pdf_page': pdf_page,
        'official_url': xvi_source_url,
        'basis': 'Vietnamese tariff column, compared visually with the original PDF; raw English column retained',
    }
source_evidence.extend({
    'target': sid, 'local_pdf': 'data/raw/TT_31-Section-XVI.pdf', 'pdf_page': page,
    'printed_page': str(page + 757), 'official_url': xvi_source_url,
    'decision': 'Remove bilingual OCR spill from normalized Vietnamese and all descendant search views',
    'local_pdf_sha256': hashlib.sha256((ROOT / 'data/raw/TT_31-Section-XVI.pdf').read_bytes()).hexdigest(),
} for sid, (_, page, _) in verified_xvi_texts.items())
display(pd.DataFrame(source_evidence))

for m in data['tt31_section_xvii_master']:
    if m['id'] == 'SXVII_HEADING4_8704':
        assert m['content']['description_vi_normalized'] in (OLD_8704, VERIFIED_8704)
        m['content']['description_vi_normalized'] = VERIFIED_8704
        for key, value in m['retrieval'].items():
            if isinstance(value, str):
                m['retrieval'][key] = value.replace(OLD_8704, VERIFIED_8704).replace(OLD_8704.casefold(), VERIFIED_8704.casefold())
        m['normalization']['verified_whitespace_correction'] = {
            'verified_on': verified_on, 'local_pdf_page': 50, 'official_pdf_page': 58,
            'official_url': VIETNAM_8704_URL, 'raw_extraction_preserved': True}
xiv_masters = {m['id']: m for m in data['tt31_section_xiv_master']}
for row in data['tt31_section_xiv']:
    m = xiv_masters[row['source_record_id']]
    vi_text = xiv_vietnamese_view(m, xiv_masters)
    row['content']['embedding_text'] = vi_text
    row['content']['sparse_text'] = vi_text
    row['content']['search_text'] = vi_text
    row['content']['fts']['text_en'] = None
    row['content']['exact_keys'] = [v for v in (
        m['hs'].get('code'), m['hs'].get('code_compact'), m['text'].get('vi')) if v]
    for ancestor in row['hierarchy']['ancestor_descriptions']:
        ancestor['en'] = None
    row['search_language'] = 'vi'
assert all(r['search_language'] == 'vi' for r in data['tt31_section_xiv'])

xvii_masters = {m['id']: m for m in data['tt31_section_xvii_master']}
for row in data['tt31_section_xvii']:
    m = xvii_masters[row['source_node_id']]
    vi_text = xvii_vietnamese_view(m)
    row['content']['embedding_text'] = vi_text
    row['content']['fts_text'] = vi_text
    row['language_profile'] = 'vi'
    m['retrieval']['search_text_vi'] = vi_text
    m['retrieval']['search_text'] = vi_text
    m['retrieval']['search_text_casefold'] = vi_text.casefold()
assert all(r['language_profile'] == 'vi' and '\nEN:' not in r['content']['embedding_text']
    for r in data['tt31_section_xvii'])

schema = read(NORMALIZED / SOURCES['tt31_section_xvi_schema'])
for m in data['tt31_section_xvi_master']:
    jsonschema.Draft202012Validator(schema).validate(m)
for key in paths:
    old_rows, new_rows = original[key], data[key]
    id_field = ('record_id' if key == 'tt31_section_xvi_master' else
        'source_record_id' if key == 'tt31_section_xiv' else
        'source_node_id' if key == 'tt31_section_xvii' else 'id')
    assert [r[id_field] for r in old_rows] == [r[id_field] for r in new_rows]
    for old, new in zip(old_rows, new_rows, strict=True):
        if key == 'tt31_section_xvi_master':
            assert old['text_en'] == new['text_en'] and old['source'] == new['source']
            if old['record_id'] in verified_xvi_texts:
                assert new['text_vi'] == verified_xvi_texts[old['record_id']][0]
            else:
                assert old['text_vi'] == new['text_vi']
        elif key == 'tt31_section_xvii_master':
            assert old['content']['description_vi_raw'] == new['content']['description_vi_raw']
            assert old['code'] == new['code'] and old['parent_id'] == new['parent_id']
        elif key == 'tt31_section_xvii':
            assert old['content']['raw_authoritative'] == new['content']['raw_authoritative']
            assert old['metadata'] == new['metadata'] and old['graph'] == new['graph']
master_lookup = {r['record_id']: r for r in data['tt31_section_xvi_master']}
assert all(r['text'] == master_lookup[r['id']]['retrieval']['embedding_text_vi'] for r in data['tt31_section_xvi'])

# Keep untouched JSONL lines byte-for-byte; no additional data/report files.
repair_rows = []
for key, path in paths.items():
    lines = path.read_text().splitlines(keepends=True)
    assert len(lines) == len(data[key])
    changed = 0
    for i, (before, after) in enumerate(zip(original[key], data[key], strict=True)):
        if before != after:
            lines[i] = json.dumps(after, ensure_ascii=False) + '\n'
            changed += 1
    if changed:
        path.write_text(''.join(lines))
    repair_rows.append({'file': str(path.relative_to(ROOT)), 'changed_rows_this_run': changed,
        'sha256': hashlib.sha256(path.read_bytes()).hexdigest()})
display(pd.DataFrame(repair_rows))
print('PASS: IDs/codes, authoritative heading wording, raw extraction and graph relationships preserved.')
print('Sections XIV/XVI/XVII searchable text: Vietnamese source fields only; bilingual raw source retained for audit.')
print('Source-only correction complete. This cell does not activate a database snapshot or change retrieval policies.')
print('The subsequent build/embedding/ingestion cells apply these inputs when that workflow is run; prior benchmark outputs remain historical.')


,target,local_pdf,pdf_page,printed_page,official_url,official_pdf_page,decision,local_pdf_sha256
0,87.04,data/raw/TT31-Section-XVII.pdf,50,1003,https://congbaocdn.chinhphu.vn/CongBaoCP/VanBa...,58,Whitespace correction only; original extracted...,00a24df2696660b5f6ad06f37d01f28d2e64b8d5743992...
1,85.06,data/raw/TT_31-Section-XVI.pdf,147,904,https://www.wcoomd.org/-/media/wco/public/glob...,6,Keep the verified Vietnamese heading; exclude ...,812b348badb5d6b57748a56551d0d1bec580cc74b2a732...
2,85.34 / Chapter 85 Note 8,data/raw/TT_31-Section-XVI.pdf,179 / 122,936 / 886,https://www.wcoomd.org/-/media/wco/public/glob...,14 / 2,Legal heading retained; add complete source No...,812b348badb5d6b57748a56551d0d1bec580cc74b2a732...
3,SXVI-T-85362099,data/raw/TT_31-Section-XVI.pdf,181,938,https://congbao.chinhphu.vn/van-ban/thong-tu-s...,NaN,Remove bilingual OCR spill from normalized Vie...,812b348badb5d6b57748a56551d0d1bec580cc74b2a732...
4,SXVI-T-85369039,data/raw/TT_31-Section-XVI.pdf,184,941,https://congbao.chinhphu.vn/van-ban/thong-tu-s...,NaN,Remove bilingual OCR spill from normalized Vie...,812b348badb5d6b57748a56551d0d1bec580cc74b2a732...
5,SXVI-T-8538,data/raw/TT_31-Section-XVI.pdf,185,942,https://congbao.chinhphu.vn/van-ban/thong-tu-s...,NaN,Remove bilingual OCR spill from normalized Vie...,812b348badb5d6b57748a56551d0d1bec580cc74b2a732...
6,SXVI-T-85389012,data/raw/TT_31-Section-XVI.pdf,186,943,https://congbao.chinhphu.vn/van-ban/thong-tu-s...,NaN,Remove bilingual OCR spill from normalized Vie...,812b348badb5d6b57748a56551d0d1bec580cc74b2a732...
7,SXVI-T-85491310,data/raw/TT_31-Section-XVI.pdf,199,956,https://congbao.chinhphu.vn/van-ban/thong-tu-s...,NaN,Remove bilingual OCR spill from normalized Vie...,812b348badb5d6b57748a56551d0d1bec580cc74b2a732...


,file,changed_rows_this_run,sha256
0,data/normalized/TT31/Section-XIV/SECTION_XIV_M...,0,9b3104316a405e6a9c65a575cfa23165ad179975d14318...
1,data/normalized/TT31/Section-XIV/SECTION_XIV_R...,0,7f007e2914c4513940aee415262d9dc31ac79c712e83ee...
2,data/normalized/TT31/Section-XVI/SECTION_XVI_M...,0,53e1c534403fa4f034be09ac6d23c9dc111ba30e64afe5...
3,data/normalized/TT31/Section-XVI/SECTION_XVI_R...,0,0c4851e8f650528372f87d954214a5d87aa7041828aafe...
4,data/normalized/TT31/Section-XVII/SECTION_XVII...,0,4c4251d222e9f672248a37224dbb24a37b054181ad8ca6...
5,data/normalized/TT31/Section-XVII/SECTION_XVII...,0,3ec65c5b598a3283a7c3265f0b4738e2e608627b207df9...


PASS: IDs/codes, authoritative heading wording, raw extraction and graph relationships preserved.
Sections XIV/XVI/XVII searchable text: Vietnamese source fields only; bilingual raw source retained for audit.
Source-only correction complete. This cell does not activate a database snapshot or change retrieval policies.
The subsequent build/embedding/ingestion cells apply these inputs when that workflow is run; prior benchmark outputs remain historical.


## 1. Audit và truy vết nguồn
TT31 ingest được nối với master; GIR combined và TT85 source là allowlist. data/verified/train và test được dùng kiểm coverage mã, không đưa vào corpus.

In [4]:
from rag.data import NORMALIZED, SOURCES

audit_result = audit()
display(pd.DataFrame(audit_result["quality"]).T[["count", "duplicate_wording"]])
display(
    pd.DataFrame(
        [
            {
                "scope_chapters": audit_result["scope_count"],
                "graph_edges": audit_result["graph_edges"],
                "broken_endpoints": len(audit_result["broken_endpoints"]),
                "gold_codes_missing": len(audit_result["gold_missing"]),
                "TT85_SHA256_match": audit_result["tt85_source_hash_matches"],
                "legal_verification": audit_result["legal_verification"],
            }
        ]
    )
)



# Runtime allowlist stays at its original paths. Historical audit files are nested.
layout = []
for section in ("XIV", "XVI", "XVII"):
    folder = NORMALIZED / "TT31" / ("Section-" + section)
    runtime_files = sorted(Path(value).name for value in SOURCES.values()
        if str(Path(value).parent) == "TT31/Section-" + section)
    supporting_files = sorted(p.name for p in (folder / "supporting").iterdir() if p.is_file())
    assert all((folder / name).is_file() for name in runtime_files)
    layout.append({"section": section, "runtime_inputs": runtime_files,
        "supporting_count": len(supporting_files), "supporting": supporting_files})
display(pd.DataFrame(layout))
print("Auxiliary exports are excluded from the ingestion allowlist.")


,count,duplicate_wording
tt31_section_xvi,3421,76
gir,115,0
tt85,100,0
tt31_section_xiv,190.0,0.0
tt31_section_xvii,1433.0,4.0


,scope_chapters,graph_edges,broken_endpoints,gold_codes_missing,TT85_SHA256_match,legal_verification
0,97,5813,0,0,True,LOCAL_METADATA_ONLY


,section,runtime_inputs,supporting_count,supporting
0,XIV,"[SECTION_XIV_GRAPH_EDGES.jsonl, SECTION_XIV_GR...",6,"[SECTION_XIV_CORRECTION_LOG.json, SECTION_XIV_..."
1,XVI,"[SECTION_XVI_GRAPH_EDGES.jsonl, SECTION_XVI_GR...",4,"[SECTION_XVI_EXACT_CODE_INDEX.csv, SECTION_XVI..."
2,XVII,"[SECTION_XVII_GRAPH_EDGES_NORMALIZED.jsonl, SE...",12,[SECTION_XVII_BENCHMARK_FINAL_CORRECTIONS.json...


Auxiliary exports are excluded from the ingestion allowlist.


## 2. Corpus và metadata thống nhất
GIR giữ source atomic và context view; TT85 giữ điều khoản, loại 5 forms khỏi RAG. Structural/context/footnote chunks giữ nguyên vai trò và chunk_type=null khi không thuộc 7 loại statutory/tariff đã chốt, tránh đổi tên footnote thành điều luật. Scope index ngoài 3 Sections chỉ hỗ trợ nhận biết phạm vi.


In [5]:
bundle = build_corpus()
records = pd.DataFrame(bundle["records"])
display(
    records.groupby(["namespace", "dataset", "section", "role"], dropna=False)
    .size()
    .rename("records")
    .reset_index()
)
display(
    records.groupby(["chunk_type", "level"], dropna=False).size().rename("records").reset_index()
)
print("Snapshot:", bundle["snapshot"], "Corpus:", len(records))
assert len(records) == 5356
assert records["id"].is_unique
assert not records["id"].str.contains("TT85-FORM-").any()
assert set(records["namespace"]) == {"hs_classification", "legal_classification"}

# Confirm corrected heading text is inside the rebuilt corpus, not just on disk.
corrected_headings = [r for r in bundle["records"] if r["section"] == "XVII"
    and r["level"] == 4 and r["code"] in ("8711", "8716")]
assert len(corrected_headings) == 2
display(pd.DataFrame([{ "id": r["id"], "code": r["code"],
    "source_text": r["source_text"], "embedding_text": r["embedding_text"],
    "input_hash": r["input_hash"]} for r in corrected_headings]))
print("Current input hashes:", {key: value for key, value in bundle["manifest"]["source_hashes"].items()
    if "Section-XVII" in key})


,namespace,dataset,section,role,records
0,hs_classification,gir,NaN,evidence,115
1,hs_classification,scope,I,scope,5
2,hs_classification,scope,II,scope,9
3,hs_classification,scope,III,scope,1
4,hs_classification,scope,IV,scope,9
5,hs_classification,scope,IX,scope,3
6,hs_classification,scope,V,scope,3
7,hs_classification,scope,VI,scope,11
8,hs_classification,scope,VII,scope,2
9,hs_classification,scope,VIII,scope,3


,chunk_type,level,records
0,chapter_note,NaN,102
1,gir_explanation,NaN,104
2,gir_rule,NaN,11
3,legal_clause,NaN,100
4,section_note,NaN,34
5,subheading_note,NaN,15
6,tariff_entry,2.0,5
7,tariff_entry,4.0,191
8,tariff_entry,6.0,645
9,tariff_entry,8.0,3480


Snapshot: 1a67df40d28f8b07a668746c Corpus: 5356


,id,code,source_text,embedding_text,input_hash
0,tt31:SXVII_HEADING4_8711,8711,Mô tô (kê cả xe gắn máy có bàn đạp (moped)) và...,Mã 87.11. Mô tô (kể cả xe gắn máy có bàn đạp (...,8c69c82729c0ec6479a8ff228df3e178c5515a9963339a...
1,tt31:SXVII_HEADING4_8716,8716,"Rơ-moóc và sơ mi rơ-moóc; xe khác, không có cơ...","Mã 87.16. Rơ-moóc và sơ mi rơ-moóc; xe khác, k...",ccd3d9b44781435654bba6a2643267b2c202bd5f5cffa0...


Current input hashes: {'TT31/Section-XVII/SECTION_XVII_RAG_INGEST.jsonl': '3ec65c5b598a3283a7c3265f0b4738e2e608627b207df94508a36bfbf72f003f', 'TT31/Section-XVII/SECTION_XVII_MASTER_NORMALIZED.jsonl': '4c4251d222e9f672248a37224dbb24a37b054181ad8ca60a2f1e222176c3f8d7', 'TT31/Section-XVII/SECTION_XVII_GRAPH_EDGES_NORMALIZED.jsonl': '5662495d6bbd4237cf82adc3044dc8120b3e14513fc4cbae9f19112dc92989e4'}


In [6]:
from rag.data import NORMALIZED, read

examples = [
    r
    for r in bundle["records"]
    if r["id"] in ["gir:GIR_3_NOTE_038", "gir:GIR_5_NOTE_016", "tt85:TT85-A9-3.a.1"]
]
display(
    pd.DataFrame(
        [
            {
                "id": r["id"],
                "source_text": r["source_text"],
                "embedding_view": r["embedding_text"],
                "locator": r["locator"],
            }
            for r in examples
        ]
    )
)

forms = read(NORMALIZED / "TT85/tt85_forms_source.json")
display(
    pd.DataFrame(
        [
            {
                "form": f["form_no"],
                "code": f["code"],
                "title": f["title"],
                "fields": len(f["fields"]),
            }
            for f in forms
        ]
    )
)
print("5 forms are a template registry, not RAG chunks; Word generation is not implemented.")
assert len(forms) == 5

,id,source_text,embedding_view,locator
0,gir:GIR_3_NOTE_038,Phân loại vào nhóm 16.02.,GIR Quy tắc 3 | GIR_EXPLANATORY | (a) Bộ thực...,"{'document': 'Phụ lục II - 6_GIR.pdf', 'pages'..."
1,gir:GIR_5_NOTE_016,(Bao bì),GIR Quy tắc 5 | GIR_EXPLANATORY | (Bao bì)\nQ...,"{'document': 'Phụ lục II - 6_GIR.pdf', 'pages'..."
2,tt85:TT85-A9-3.a.1,a.1) Cơ quan hải quan thực hiện tiếp nhận và k...,"85/2026/TT-BTC | Điều 9: Hồ sơ, thủ tục đối vớ...","{'document': '85/2026/TT-BTC', 'article': 9, '..."


,form,code,title,fields
0,01,01/PYCPT/2026,"Phiếu yêu cầu phân tích hàng hóa xuất khẩu, nh...",14
1,02,02/PTNYCPT/2026,Phiếu tiếp nhận yêu cầu phân tích hàng hóa xuấ...,9
2,03,03/BBTLMHH/2026,Biên bản trả lại mẫu đã tiếp nhận yêu cầu phân...,11
3,04,04/TBKQPTPL/2026,"Thông báo về kết quả phân tích, phân loại đối ...",15
4,05,05/PGTL/2026,"Phiếu ghi số, ngày văn bản, chứng từ thuộc hồ ...",6


5 forms are a template registry, not RAG chunks; Word generation is not implemented.


## 3. BGE-M3 dense + sparse
Pin model/tokenizer commit. Corpus input không được truncate im lặng. Checkpoint theo batch/input hash giúp tái chạy nhanh. CPU FP32 và sparse là cấu hình đã chọn cho experiment; dense là 1.024 chiều, sparse là token weights của model.

In [7]:
embedder = Embedder(cfg)
started = time.perf_counter()
with capture_output() as embedding_progress:
    embedding_manifest = embedder.corpus(bundle)
manifest, vectors, sparse = load_embeddings(bundle)
display(
    pd.DataFrame(
        [
            {
                "records": len(vectors),
                "dimension": vectors.shape[1],
                "max_input_tokens": manifest["max_tokens"],
                "sparse_nonzero": sum(len(w) for w in sparse),
                "elapsed_seconds": round(time.perf_counter() - started, 2),
                "model_revision": manifest["profile"]["revision"],
                "profile": manifest["profile_id"],
            }
        ]
    )
)
np.testing.assert_allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-5)
assert len(sparse) == len(vectors)
assert np.isfinite(vectors).all()

/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

Fetching 22 files: 100%|██████████| 22/22 [00:00<00:00, 5339.66it/s]

,records,dimension,max_input_tokens,sparse_nonzero,elapsed_seconds,model_revision,profile
0,5356,1024,523,217109,1.08,5617a9f61b028005a4858fdac845db406aefb181,3a2ce6e6d726a0f053748717


## 4. PostgreSQL ingestion và snapshot activation
Chỉ snapshot đã đủ nguồn, vectors và graph mới được chuyển active. Snapshot cũ được giữ để rollback. `.env` không xuất hiện trong output.

In [8]:
conn = connect()
migrate(conn)
display(
    pd.DataFrame(
        [conn.execute("SELECT current_database() AS database, version() AS server").fetchone()]
    )
)
APPLY = True
print("Apply to the displayed RAG database:", APPLY)

,database,server
0,hs_rag,PostgreSQL 17.6 (Debian 17.6-1.pgdg12+1) on aa...


Apply to the displayed RAG database: True


In [9]:
ingestion_result = ingest(conn, bundle, manifest, vectors, sparse, apply=APPLY)
display(pd.DataFrame([ingestion_result.get("counts", ingestion_result.get("expected"))]))
if APPLY:
    before = counts(conn, bundle["snapshot"], manifest["profile_id"])
    ingest(conn, bundle, manifest, vectors, sparse, apply=True)
    after = counts(conn, bundle["snapshot"], manifest["profile_id"])
    assert before == after
    print("Idempotency: PASS")
    display(pd.DataFrame([conn.execute("SELECT * FROM rag.active").fetchone()]))

,chunks,embeddings,nodes,edges
0,5356,5356,5521,6655


Idempotency: PASS


,singleton,snapshot,profile
0,True,1a67df40d28f8b07a668746c,3a2ce6e6d726a0f053748717


## 5. Kiểm chứng cuối
Counts, FK integrity, vector norms và sparse coverage đều được kiểm trực tiếp. Notebook 02 dùng snapshot/profile active này; không cần giữ kernel hiện tại.

In [10]:
if APPLY:
    check = conn.execute(
        """SELECT count(*) AS sparse_postings FROM rag.sparse_postings
        WHERE snapshot=%s AND profile=%s""",
        (bundle["snapshot"], manifest["profile_id"]),
    ).fetchone()
    assert check["sparse_postings"] == sum(len(w) for w in sparse)
    print(
        "Knowledge base ready:", {"snapshot": bundle["snapshot"], "records": len(vectors), **check}
    )
conn.close()

Knowledge base ready: {'snapshot': '1a67df40d28f8b07a668746c', 'records': 5356, 'sparse_postings': 217109}


## Definition graph audit for retrieval
The active knowledge base already contains Notes and definition edges. Derive cited search views from that active snapshot; do not re-ingest gold or change atomic source wording. Search views are built in memory for BM25 and optional evidence-aware reranking. Existing BGE dense/sparse vectors remain valid. Notebook 02 benchmarks and freezes this retrieval policy.

In [11]:
import pandas as pd
from IPython.display import display

from rag.db import connect
from rag.retrieval import Retriever

with connect() as audit_connection:
    definition_audit = Retriever(audit_connection)
    definition_views = definition_audit.definition_views()
    active_before = audit_connection.execute("SELECT * FROM rag.active").fetchone()
    display(pd.DataFrame([{"section": view["record"]["section"],
        "heading": view["record"]["heading"], "ahtn8": view["record"]["code"],
        "source_ids": [x["id"] for x in view["citations"]],
        "graph_paths": view["paths"]} for view in definition_views.values()]))
    assert all(view["record"]["source_text"] == definition_audit.exact(view["record"]["code"])[0]["source_text"]
        for view in definition_views.values())
    assert audit_connection.execute("SELECT * FROM rag.active").fetchone() == active_before
    print("Source-backed definition views:", len(definition_views), "Active snapshot:", active_before)


,section,heading,ahtn8,source_ids,graph_paths
0,XIV,7113,71132090,[tt31:XIV_CH71_NOTE_09],"[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relatio..."
1,XIV,7113,71132010,[tt31:XIV_CH71_NOTE_09],"[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relatio..."
2,XIV,7113,71131990,[tt31:XIV_CH71_NOTE_09],"[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relatio..."
3,XIV,7113,71131910,[tt31:XIV_CH71_NOTE_09],"[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relatio..."
4,XIV,7113,71131190,[tt31:XIV_CH71_NOTE_09],"[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relatio..."
5,XIV,7113,71131110,[tt31:XIV_CH71_NOTE_09],"[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relatio..."
6,XIV,7114,71142000,[tt31:XIV_CH71_NOTE_10],"[{'note_id': 'tt31:XIV_CH71_NOTE_10', 'relatio..."
7,XIV,7114,71141900,[tt31:XIV_CH71_NOTE_10],"[{'note_id': 'tt31:XIV_CH71_NOTE_10', 'relatio..."
8,XIV,7114,71141100,[tt31:XIV_CH71_NOTE_10],"[{'note_id': 'tt31:XIV_CH71_NOTE_10', 'relatio..."
9,XIV,7117,71179099,"[tt31:XIV_CH71_NOTE_11, tt31:XIV_CH71_NOTE_09]","[{'note_id': 'tt31:XIV_CH71_NOTE_11', 'relatio..."


Source-backed definition views: 49 Active snapshot: {'singleton': True, 'snapshot': '1a67df40d28f8b07a668746c', 'profile': '3a2ce6e6d726a0f053748717'}
